In [1]:
partition = 200

In [2]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [4]:
import random
from itertools import product
import sys

log_path = f"logs{partition}.txt"
tested_configs = load_tested_configs(log_path)

n_tree_values = [5, 10, 20, 50, 100]
tree_depth_values = [6, 7, 8, 9, 10]
hidden_dim = [1024, 768]
batch_size_values = [256, 512]
tree_feature_rates = [0.0, 0.1, 0.2, 0.3, 0.4]
feat_dropouts = [0.0, 0.1, 0.2, 0.3]
lrs = [0.001, 0.01]

n_iter = 150
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))

best_acc = 0

available_configs = [cfg for cfg in param_space if cfg not in tested_configs]
sampled_configs = random.sample(available_configs, min(n_iter, len(available_configs)))
i = 1

for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in sampled_configs:
    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")

    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
    i = i+1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(f"Best accuracy: {best_acc}")



Running: n_tree=10, t_depth=6, hd=1024, batch_size=256, feature_rate=0.3, dropout=0.2, lr=0.001
1 / 100
Use gtd200 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [00:58<00:00,  6.81it/s]



Best Accuracy: 0.827381

Running: n_tree=5, t_depth=6, hd=768, batch_size=512, feature_rate=0.3, dropout=0.0, lr=0.001
2 / 100
Use gtd200 dataset
Patience: 100


Training Epochs:  83%|████████▎ | 331/400 [00:16<00:03, 19.61it/s]


Early stopping at epoch 332

Best Accuracy: 0.772619

Running: n_tree=20, t_depth=8, hd=768, batch_size=256, feature_rate=0.1, dropout=0.0, lr=0.001
3 / 100
Use gtd200 dataset
Patience: 100


...


Best Accuracy: 0.778571

Running: n_tree=10, t_depth=7, hd=1024, batch_size=512, feature_rate=0.2, dropout=0.2, lr=0.01
149 / 100
Use gtd200 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [00:35<00:00, 11.36it/s]
/opt/conda/lib/python3.11/site-packages/torch/nn/init.py:452: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")



Best Accuracy: 0.860714

Running: n_tree=20, t_depth=8, hd=1024, batch_size=256, feature_rate=0.0, dropout=0.3, lr=0.01
150 / 100
Use gtd200 dataset
Patience: 100


Training Epochs:  25%|██▌       | 100/400 [00:29<01:28,  3.37it/s]

Early stopping at epoch 101

Best Accuracy: 0.033333

Best hyperparameter configuration:
{'n_tree': 100, 'tree_depth': 10, 'batch_size': 512, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'lr': 0.01}
Best accuracy: 0.9154761904761904


In [5]:
#Running: n_tree=100, t_depth=10, hd=1024, batch_size=512, feature_rate=0.3, dropout=0.1, lr=0.01



In [6]:

"""
========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd200
  Hidden Dim: 768
  n_tree: 100, tree_depth: 10, tree_feature_rate: 0.1
  Batch size: 512, Dropout: 0.1, LR: 0.01

Best Accuracy: 0.9011
Weighted Precision: 0.9030, Recall: 0.9011, F1 Score: 0.8994, ROCAUC: 0.9971
Macro Precision: 0.9030, Recall: 0.9011, F1 Score: 0.8994, ROCAUC: 0.9971
Micro Precision: 0.9011, Recall: 0.9011, F1 Score: 0.9011, ROCAUC: 0.9976
"""

'\n\n========== Final Test Evaluation ==========\nModel Parameters:\n  Dataset: gtd200\n  Hidden Dim: 1024\n  n_tree: 100, tree_depth: 10, tree_feature_rate: 0.3\n  Batch size: 512, Dropout: 0.1, LR: 0.01\n\nBest Accuracy: 0.8878\nWeighted Precision: 0.8946, Recall: 0.8878, F1 Score: 0.8856, ROCAUC: 0.9971\nMacro Precision: 0.8946, Recall: 0.8878, F1 Score: 0.8856, ROCAUC: 0.9971\nMicro Precision: 0.8878, Recall: 0.8878, F1 Score: 0.8878, ROCAUC: 0.9976\n'

In [7]:
sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()

Use gtd200 dataset
Patience: 300


Training Epochs:   3%|▎         | 50/1500 [00:45<21:43,  1.11it/s]

[Epoch 50] Train Loss: 0.5884, Eval Loss: 0.6320, Eval Accuracy: 0.8631


Training Epochs:   3%|▎         | 51/1500 [00:46<21:58,  1.10it/s]

Training Epochs:   7%|▋         | 100/1500 [01:29<20:12,  1.15it/s]

...

[Epoch 900] Train Loss: 0.3204, Eval Loss: 0.4137, Eval Accuracy: 0.9119


Training Epochs:  63%|██████▎   | 950/1500 [13:32<07:48,  1.17it/s]

[Epoch 950] Train Loss: 0.3190, Eval Loss: 0.4129, Eval Accuracy: 0.9083


Training Epochs:  64%|██████▍   | 966/1500 [13:47<07:37,  1.17it/s]

Early stopping at epoch 967
Evaluating on test set with best model...


In [8]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.92      0.93      0.93        60
        African National Congress (South Africa)       1.00      1.00      1.00        60
                                Al-Qaida in Iraq       0.66      0.82      0.73        60
        Al-Qaida in the Arabian Peninsula (AQAP)       0.90      0.88      0.89        60
                                      Al-Shabaab       1.00      0.98      0.99        60
             Basque Fatherland and Freedom (ETA)       1.00      0.98      0.99        60
                                      Boko Haram       0.95      0.88      0.91        60
  Communist Party of India - Maoist (CPI-Maoist)       0.97      0.95      0.96        60
       Corsican National Liberation Front (FLNC)       0.97      0.98      0.98        60
                       Donetsk People's Republic       0.98      0.98      0.98        60
Farabundo

In [9]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [10]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true